In [3]:
import surprise
import pandas as pd
import numpy as np
import os
os.chdir("C:/Users/PGCP-AI/ML/MachineLearning/Cases_Rec_Sys/filmtrust")

In [4]:
rating = pd.read_csv('ratings.txt',sep=' ',names = ['uid','Iid','rating'])
rating

,uid,Iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0
...,...,...,...
35492,1508,84,3.5
35493,1508,17,4.0
35494,1508,669,1.0
35495,1508,686,2.5


In [5]:
lowest_rating = rating['rating'].min()
highest_rating = rating['rating'].max()
lowest_rating,highest_rating

(0.5, 4.0)

In [6]:
reader = surprise.Reader(rating_scale=(lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(rating,reader)

In [7]:
similarity_options = {'name':'cosine','user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset()) 
#calculates expected rating for all the users 

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [8]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [12]:
param_grid ={'k':[25,40,50,65,75],'user_based':[True]}
kfold= KFold(n_splits=5,shuffle=True,random_state=26)
gs=GridSearchCV(surprise.KNNBasic,param_grid,measures=['rmse','mae'],cv=kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [13]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.8642083344321752
{'k': 65, 'user_based': True}


In [20]:
user = 50
u_iid = rating[rating['uid']==user]['Iid'].unique()
iids = rating['Iid'].unique()
print('List of items rated by user:',u_iid)
print('No. of items rated by user {0} : {1} '.format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(iids,u_iid)
print("Items not rated by the user or those items for which the expected ratings are to be predicted",iids_to_predict)


List of items rated by user: [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]
No. of items rated by user 50 : 39 
Items not rated by the user or those items for which the expected ratings are to be predicted [  14   15   16 ... 2069 2070 2071]


In [21]:
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
df

,iid,est_rating
0,14,1.024911
1,15,2.301082
2,16,3.365656
3,18,3.475089
4,19,2.950177
...,...,...
2027,2067,3.000000
2028,2068,2.500000
2029,2069,2.500000
2030,2070,3.000000
